# Day 62 - Mixture of Experts and routing

Day 53 measured where a transformer's parameters live: two thirds of them in the
feed-forward block, and every token pays for all of them. MoE replaces that one FFN
with E copies and sends each token to the best k - so the parameters scale with E and
the arithmetic per token with k.

The price is everything else in this notebook: the router has to learn to spread the
tokens, the buffers are fixed size, and the experts live on different GPUs.

In [ ]:
import numpy as np
from moe import (ffn_vs_moe, width_equivalent, route, expert_loads, balance_stats,
                 make_task, train_moe, dense_baseline, aux_loss, capacity_drop,
                 ep_simulation, ep_vs_tp, deepep_model)

acc = ffn_vs_moe()
for k in ('dense_params', 'moe_params', 'active_params', 'param_ratio', 'flop_ratio'):
    print(f"{k:16} {acc[k]:,.2f}")
print('model total / active (B):', round(acc['model_total']), round(acc['model_active']))

dense_params     176,160,768.00
moe_params       11,450,449,920.00
active_params    528,482,304.00
param_ratio      65.00
flop_ratio       3.00
model total / active (B): 698 32


## 1. A routed MoE, trained from scratch

Tokens whose right answer depends on which region of space they fall in - the
structure an MoE is supposed to discover. With no balancing, the router collapses.

In [ ]:
task = make_task()
dense = dense_baseline(task)
plain = train_moe(task)
print('dense loss', round(dense, 4), ' MoE loss', round(plain['loss'], 4))
print('tokens per expert', sorted(plain['counts'].tolist(), reverse=True))
print(balance_stats(plain['counts']))

dense loss 0.0363  MoE loss 0.023
tokens per expert [579, 554, 550, 475, 461, 405, 352, 323, 293, 84, 14, 3, 1, 1, 1, 0]
{'max_over_mean': 2.26171875, 'min_over_mean': 0.0, 'dead': 1, 'entropy': 0.8121028305740676, 'cv': 0.8840981633155505, 'ideal_share': 0.0625}


## 2. Two ways to spread the tokens

An auxiliary loss (Switch / GShard), or a per-expert bias used for selection only
(DeepSeek-V3's loss-free scheme).

In [ ]:
bal = train_moe(task, aux=0.3)
free = train_moe(task, bias_rate=0.001)
for name, r in [('none', plain), ('aux loss', bal), ('bias', free)]:
    b = r['balance']
    print(f"{name:9} loss {r['loss']:.4f}  busiest/avg {b['max_over_mean']:.2f}"
          f"  dead {b['dead']}  entropy {b['entropy']:.3f}  purity {r['purity']:.3f}")

none      loss 0.0230  busiest/avg 2.26  dead 1  entropy 0.812  purity 0.641
aux loss  loss 0.0241  busiest/avg 1.98  dead 0  entropy 0.922  purity 0.645
bias      loss 0.0257  busiest/avg 1.05  dead 0  entropy 1.000  purity 0.568


## 3. Capacity: fixed buffers mean dropped tokens

In [ ]:
for f in (1.0, 1.25, 1.5, 2.0):
    a = capacity_drop(plain['counts'], task['X'].shape[0], 2, 16, f)
    b = capacity_drop(free['counts'], task['X'].shape[0], 2, 16, f)
    print(f"factor {f:4} capacity {a['capacity']:4d}  dropped {a['drop_rate']:6.2%}"
          f" (unbalanced) vs {b['drop_rate']:6.2%} (balanced)  padding +{a['padding']:.0%}")

factor  1.0 capacity  256  dropped 41.21% (unbalanced) vs  1.05% (balanced)  padding +0%
factor 1.25 capacity  320  dropped 27.81% (unbalanced) vs  0.00% (balanced)  padding +25%
factor  1.5 capacity  384  dropped 17.58% (unbalanced) vs  0.00% (balanced)  padding +50%
factor  2.0 capacity  512  dropped  3.59% (unbalanced) vs  0.00% (balanced)  padding +100%


## 4. Expert parallelism, and the all-to-all

The experts are split across GPUs, so every token travels to its expert and back.
The router's imbalance becomes idle GPUs, because a collective is synchronous.

In [ ]:
ep = ep_simulation(plain['counts'], n_gpu=4, d_model=4096)
epb = ep_simulation(free['counts'], n_gpu=4, d_model=4096)
print('tokens per GPU, unbalanced', ep['gpu_tokens'].tolist(),
      ' busiest/avg', round(ep['max_over_mean'], 2),
      ' idle', f"{ep['idle_share']:.0%}")
print('tokens per GPU, balanced  ', epb['gpu_tokens'].tolist(),
      ' busiest/avg', round(epb['max_over_mean'], 2))
print()
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in ep_vs_tp().items()
       if k != 'note'})
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in deepep_model().items()})

tokens per GPU, unbalanced [1582, 798, 1361, 355]  busiest/avg 1.54  idle 35%
tokens per GPU, balanced   [1022, 1001, 1032, 1041]  busiest/avg 1.02

{'tp_mb': 58.72, 'ep_mb': 117.44, 'ratio': 2.0, 'k': 2, 'weights_per_gpu_gb': 2.82}
{'tokens': 4096, 'k': 8, 'moved': 32768, 'mb_bf16': 469.762, 'mb_fp8': 234.881, 'cross_share': 0.875, 'ms_bf16': 8.588, 'ms_fp8': 4.294, 'speedup': 2.0}


## 5. Tests

In [ ]:
acc = ffn_vs_moe()
assert acc['param_ratio'] > 10 * acc['flop_ratio']
assert plain['loss'] < dense
assert bal['balance']['max_over_mean'] < plain['balance']['max_over_mean']
assert free['balance']['dead'] == 0
dp = deepep_model()
assert abs(dp['speedup'] - 2.0) < 1e-9        # fp8 dispatch halves the bytes
print('all assertions passed')

all assertions passed
